# Preprocesamiento de Datos - Proyecto HTA (SIS)

## Dataset: Prestaciones de salud asociadas a los asegurados con Hipertensión Arterial (SIS)

### Origen y ficha técnica

| Campo | Detalle |
|---|---|
| **Título original** | Prestaciones de salud asociadas a los asegurados con Hipertensión Arterial - [SIS] |
| **Entidad / fuente** | Seguro Integral de Salud (SIS) – Oficina General de Tecnología de la Información |
| **Portal** | [Plataforma Nacional de Datos Abiertos](https://www.datosabiertos.gob.pe/dataset/prestaciones-de-salud-asociadas-los-asegurados-con-hipertensi%C3%B3n-arterial-sis) |
| **Licencia** | Open Data Commons Attribution License (ODC-BY) |
| **Creación / última actualización** | 2024-06-06 / 2025-08-29 (actualización trimestral, versión 1.0) |
| **Formato y cobertura** | CSV · Perú (nacional) · español |
| **Contacto** | soporte@sis.gob.pe |

### ¿Qué registra este dataset?

El SIS es la aseguradora pública más grande del Perú: cubre financieramente las atenciones de más de 27.2 millones de asegurados (74% de la población) y tiene convenios con más de 8,600 establecimientos de salud (IPRESS) del Ministerio de Salud y de los gobiernos regionales.

Este dataset reúne **las prestaciones de salud de los asegurados con Hipertensión Arterial (HTA)**. Un afiliado se considera hipertenso si el SIS encontró **diagnósticos definitivos de HTA** en sus atenciones **desde 2018 hasta la fecha de corte**. De cada atención se publica el detalle de lo que se registró en el **FUA (Formato Único de Atención)**: datos del afiliado (sexo, edad), del establecimiento (ubigeo, departamento, provincia, distrito, nivel), de la atención (identificador, fecha, servicio prestacional), los diagnósticos y los consumos (medicamentos, procedimientos e insumos con cantidad entregada y valor neto).

> ⚠️ El propio SIS advierte que los FUA los registra cada establecimiento y que **pueden contener datos no plausibles**. Por eso este cuaderno dedica una parte importante a validar y limpiar.





### Diferencias entre la documentación y los archivos

- La documentación dice que la información se publica **"desde el año 2023"**, pero los archivos traen atenciones desde **enero de 2022**.
- El SIS identificó a los hipertensos con atenciones desde **2018**, pero los archivos solo incluyen **2022-2024**: **no tenemos el historial anterior** ni sabemos desde cuándo es hipertenso cada paciente.
- Se menciona el "nombre de consumo", pero los archivos de consumos solo traen **códigos** (`COD_MEDICAMENTO`, `COD_PROCEDIMIIENTO`, `COD_INSUMO`); sin un catálogo externo no se sabe qué medicamento o procedimiento es cada código (por ejemplo, cuáles son antihipertensivos).
- Los metadatos no incluyen un **diccionario de columnas**; el de la siguiente sección es una interpretación a partir de los nombres y de los datos observados.

## Objetivo

Construir, a partir de las atenciones registradas por el SIS, una tabla **por paciente** que permita predecir si un paciente hipertenso presentará una **complicación grave** en el año siguiente al período observado.

- **Unidad de análisis:** el paciente (`CODIGO_ANONIMIZADO`), no la atención.
- **Variables (X):** lo ocurrido en la ventana de observación 2022-2023 (atenciones, consumos, presión arterial, comorbilidades).
- **Variable objetivo (y):** Sin complicación (0) / Con complicación (1) en 2024. Se considera complicación cualquiera de estos grupos de diagnósticos CIE-10 (definitivos o repetidos) o el fallecimiento:

| Grupo | Códigos CIE-10 (prefijo de 3 caracteres) |
|---|---|
| ACV / AIT | `I60`-`I64`, `G45` |
| Infarto / IAM | `I21`-`I23` |
| Insuficiencia cardíaca | `I50` |
| Enfermedad renal | `N17`-`N19` |
| Diabetes | `E10`-`E14` |
| Dislipidemia | `E78` |
| Obesidad | `E66` |
| Fibrilación auricular | `I48` |
| Cardiopatía isquémica | `I20`, `I24`, `I25` |
| Fallecimiento | `DESTINO_ASEGURADO == 'Fallecido'` |

### Idea clave del cuaderno
Las tablas tienen **una fila por atención**, pero se quiere **predecir por paciente**. Además, no se pueden usar datos del futuro para predecir el futuro:

```
|-------- 2022 y 2023 --------|---- 2024 ----|
 Variables (X): lo que ya pasó   Objetivo (y): ¿tuvo complicación?
```

Este cuaderno es exploratorio: cada análisis se presenta como (1) qué se quiere saber y por qué, (2) el código y (3) qué implica el resultado para la limpieza y la unión de las tablas.

## Inicializar

In [1]:
import sys
!{sys.executable} -m pip install imbalanced-learn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
RUTA = ""            # carpeta de los CSV. "" = la misma carpeta del cuaderno.
                     # Si están en otra: RUTA = "C:/Users/ROG STRIX/Desktop/IA/ProyectoIA/"
SEED = 42            # semilla para que los resultados se repitan (fija el azar de particiones y remuestreos)

## Importar datasets
Se cargan los 5 CSV tal como se publicaron, antes de tocar nada, para partir de los datos originales. Los códigos de medicamento y procedimiento se leen como TEXTO (`dtype=str`) porque son identificadores, no cantidades: si se leyeran como número se perderían los ceros a la izquierda (p. ej. `'02642'` -> `2642`) y dos códigos distintos podrían confundirse. Insumos no lo necesita (`COD_INSUMO` ya se lee como entero). Nota: `COD_PROCEDIMIIENTO` (doble "I") es como se llama la columna en el archivo original; no es un error de este cuaderno.

Al final se imprime el tamaño de cada tabla: es necesario confirmar que la carga fue completa (filas y columnas esperadas) antes de seguir; si alguna cifra no coincide con el archivo, hubo un problema de lectura.

In [4]:
fuas           = pd.read_csv(RUTA + '1. HTA_FUAS.csv')
diagnosticos   = pd.read_csv(RUTA + '2. HTA_Diagnosticos.csv')
medicamentos   = pd.read_csv(RUTA + '3. HTA_Medicamentos.csv', dtype={'COD_MEDICAMENTO': str})
procedimientos = pd.read_csv(RUTA + '4. HTA_Procedimientos.csv', dtype={'COD_PROCEDIMIIENTO': str})
insumos        = pd.read_csv(RUTA + '5. HTA_Insumos.csv')

print("FUAS:", fuas.shape)
print("Diagnósticos:", diagnosticos.shape)
print("Medicamentos:", medicamentos.shape)
print("Procedimientos:", procedimientos.shape)
print("Insumos:", insumos.shape)

FUAS: (24286707, 21)
Diagnósticos: (38079013, 5)
Medicamentos: (39352038, 7)
Procedimientos: (53027090, 7)
Insumos: (15767437, 7)


## Exploración global de los 5 datasets

### Los 5 archivos

El dataset se publica como **5 archivos CSV** independientes (originalmente 5 tablas separadas), que hay que unir para construir una sola tabla de modelado:

| # | Archivo | Tabla | Qué contiene | Una fila es... | Filas | Columnas |
|---|---|---|---|---|---|---|
| 1 | `1. HTA_FUAS.csv` | FUAS | Datos del paciente, del establecimiento y de la atención (fecha, edad, presión arterial, hospitalización, destino) | una atención | 24,286,707 | 21 |
| 2 | `2. HTA_Diagnosticos.csv` | Diagnósticos | Códigos CIE-10 registrados en cada atención y su tipo (definitivo, repetido, presuntivo) | un diagnóstico de una atención | 38,079,013 | 5 |
| 3 | `3. HTA_Medicamentos.csv` | Medicamentos | Medicamentos entregados, cantidad y valor neto | una entrega de medicamento | 39,352,038 | 7 |
| 4 | `4. HTA_Procedimientos.csv` | Procedimientos | Procedimientos realizados (laboratorio, imágenes, etc.), cantidad y valor neto | un procedimiento | 53,027,090 | 7 |
| 5 | `5. HTA_Insumos.csv` | Insumos | Insumos y dispositivos entregados, cantidad y valor neto | una entrega de insumo | 15,767,437 | 7 |

En total son **≈170.5 millones de filas**.

In [5]:
dataframes = {
    "FUAS": fuas,
    "Diagnosticos": diagnosticos,
    "Medicamentos": medicamentos,
    "Procedimientos": procedimientos,
    "Insumos": insumos
}

#### Primer vistazo a cada tabla
Para cada tabla se revisa (1) el tamaño, (2) el tipo de cada columna y (3) dos filas de ejemplo. Es necesario saberlo para confirmar que los tipos son los esperados (p. ej. que las fechas llegan como enteros AAAAMMDD y no como fechas, y los códigos como texto), detectar columnas que no aportan información y entender qué significa cada campo antes de profundizar. `display` es una función propia de Jupyter que muestra el DataFrame con formato de tabla; `head(2)` alcanza para ver el formato sin llenar la pantalla.

In [6]:
for nombre, df in dataframes.items():
    total_filas = len(df)
    print(f"\nRESULTADOS DE {nombre.upper()}")
    print(f"Total de filas: {total_filas:,} y total de columnas: {df.shape[1]}")
    info_df = pd.DataFrame({'Tipo': df.dtypes})
    print("\nDetalle de columnas:")
    print(info_df)
    print("\n• Muestra de datos:")
    display(df.head(2))


RESULTADOS DE FUAS
Total de filas: 24,286,707 y total de columnas: 21

Detalle de columnas:
                             Tipo
FECHA_CORTE                 int64
ULTIMO_MES_CONSUMOS         int64
CODIGO_ANONIMIZADO            str
SEXO                          str
ID_REGISTRO_REL             int64
FECHA_ATENCION              int64
EDAD                        int64
UBIGEO                      int64
DEPARTAMENTO                  str
PROVINCIA                     str
DISTRITO                      str
NIVEL_EESS                  int64
CODIGO_SERV_PRESTACIONAL    int64
SERVICIO_PRESTACIONAL         str
DIAS_HOSP                 float64
PRES_ART_SISTOLICA        float64
PRES_ART_DIASTOLICA       float64
TIPO_PERSONAL_SALUD           str
FECATE_POST_FECFED            str
ES_CAPITA                     str
DESTINO_ASEGURADO             str

• Muestra de datos:


,FECHA_CORTE,ULTIMO_MES_CONSUMOS,CODIGO_ANONIMIZADO,SEXO,ID_REGISTRO_REL,FECHA_ATENCION,EDAD,UBIGEO,DEPARTAMENTO,PROVINCIA,...,NIVEL_EESS,CODIGO_SERV_PRESTACIONAL,SERVICIO_PRESTACIONAL,DIAS_HOSP,PRES_ART_SISTOLICA,PRES_ART_DIASTOLICA,TIPO_PERSONAL_SALUD,FECATE_POST_FECFED,ES_CAPITA,DESTINO_ASEGURADO
0,20250829,202412,1C6B67EF3416A1FABDA6C7F2E6883F79F1A45EC482A4EC...,FEMENINO,24668778,20221021,63,21801,ÁNCASH,SANTA,...,2,62,ATENCIÓN POR EMERGENCIA,NaN,120.0,60.0,MEDICO,SI,N,Alta
1,20250829,202412,420D8A2F0CEEED8BA6B6AEC449F9BEC19AC4544F2A6E2B...,FEMENINO,24668787,20220917,71,21801,ÁNCASH,SANTA,...,2,56,CONSULTA EXTERNA,NaN,NaN,NaN,MEDICO,NO,N,Citado



RESULTADOS DE DIAGNOSTICOS
Total de filas: 38,079,013 y total de columnas: 5

Detalle de columnas:
                      Tipo
FECHA_CORTE          int64
ULTIMO_MES_CONSUMOS  int64
ID_REGISTRO_REL      int64
CODDIA                 str
TIPO_DIAGNOSTICO       str

• Muestra de datos:


,FECHA_CORTE,ULTIMO_MES_CONSUMOS,ID_REGISTRO_REL,CODDIA,TIPO_DIAGNOSTICO
0,20250829,202412,33102263,E119,REPETIDO
1,20250829,202412,33102279,N159,DEFINITIVO



RESULTADOS DE MEDICAMENTOS
Total de filas: 39,352,038 y total de columnas: 7

Detalle de columnas:
                        Tipo
FECHA_CORTE            int64
ULTIMO_MES_CONSUMOS    int64
ID_REGISTRO_REL        int64
CODDIA                   str
COD_MEDICAMENTO          str
CANTIDAD_ENTREGADA     int64
VALOR_NETO           float64

• Muestra de datos:


,FECHA_CORTE,ULTIMO_MES_CONSUMOS,ID_REGISTRO_REL,CODDIA,COD_MEDICAMENTO,CANTIDAD_ENTREGADA,VALOR_NETO
0,20250829,202412,55631544,M545,02642,3,0.0
1,20250829,202412,55631827,G632,01532,30,0.0



RESULTADOS DE PROCEDIMIENTOS
Total de filas: 53,027,090 y total de columnas: 7

Detalle de columnas:
                        Tipo
FECHA_CORTE            int64
ULTIMO_MES_CONSUMOS    int64
ID_REGISTRO_REL        int64
CODDIA                   str
COD_PROCEDIMIIENTO       str
CANTIDAD_ENTREGADA     int64
VALOR_NETO           float64

• Muestra de datos:


,FECHA_CORTE,ULTIMO_MES_CONSUMOS,ID_REGISTRO_REL,CODDIA,COD_PROCEDIMIIENTO,CANTIDAD_ENTREGADA,VALOR_NETO
0,20250829,202412,64189371,H521,85027,1,7.08
1,20250829,202412,64189371,H521,86592,1,4.34



RESULTADOS DE INSUMOS
Total de filas: 15,767,437 y total de columnas: 7

Detalle de columnas:
                        Tipo
FECHA_CORTE            int64
ULTIMO_MES_CONSUMOS    int64
ID_REGISTRO_REL        int64
CODDIA                   str
COD_INSUMO             int64
CANTIDAD_ENTREGADA     int64
VALOR_NETO           float64

• Muestra de datos:


,FECHA_CORTE,ULTIMO_MES_CONSUMOS,ID_REGISTRO_REL,CODDIA,COD_INSUMO,CANTIDAD_ENTREGADA,VALOR_NETO
0,20250829,202512,36118554,J961,15334,3,3.90
1,20250829,202512,36118554,J961,19719,3,9.72


Se observa que las fechas (`FECHA_ATENCION`, `FECHA_CORTE`) llegan como enteros AAAAMMDD, por lo tanto habrá que convertirlas a fecha si se necesitan diferencias de días; para comparar rangos basta con el entero. `UBIGEO`, `NIVEL_EESS` y `CODIGO_SERV_PRESTACIONAL` son numéricos pero representan categorías, así que no deben tratarse como cantidades. `FECHA_CORTE` y `ULTIMO_MES_CONSUMOS` parecen constantes, lo que se confirma más adelante. Los valores de `TIPO_DIAGNOSTICO` aparecen alineados con espacios a la derecha, lo que anticipa que habrá que limpiarlos con `.str.strip()` antes de filtrar.

### Diccionario de columnas

> Los metadatos del SIS no traen diccionario de columnas. Las descripciones marcadas con **(\*)** son **inferidas** del nombre de la columna y de los valores observados; conviene validarlas con el SIS antes de usarlas en el modelo.

**Columnas comunes (aparecen en los 5 archivos)**

| Columna | Tipo | Descripción |
|---|---|---|
| `FECHA_CORTE` | int (AAAAMMDD) | Fecha de corte de la extracción. En FUAS siempre vale 20250829 (coincide con la última actualización), por lo que no aporta información. |
| `ULTIMO_MES_CONSUMOS` | int (AAAAMM) | Último mes de consumos incluido en el corte (\*). En FUAS siempre vale 202412. |
| `ID_REGISTRO_REL` | int | Identificador de la atención (FUA). Es la **llave de unión** entre las 5 tablas. |

**1. FUAS (una fila por atención)**

| Columna | Tipo | Descripción |
|---|---|---|
| `CODIGO_ANONIMIZADO` | texto (hash) | Código anonimizado del asegurado. Identifica al **paciente**. |
| `SEXO` | texto | Sexo del afiliado. |
| `FECHA_ATENCION` | int (AAAAMMDD) | Fecha de la atención. Rango observado: 2022-01-01 a 2024-12-31. |
| `EDAD` | int | Edad del afiliado en años (\*). Rango observado: 14 a 119. |
| `UBIGEO`, `DEPARTAMENTO`, `PROVINCIA`, `DISTRITO` | int / texto | Ubicación geográfica del establecimiento de atención. El ubigeo es un código, no una cantidad. |
| `NIVEL_EESS` | int | Nivel del establecimiento de salud (\*). Valores observados: 0 a 3. |
| `CODIGO_SERV_PRESTACIONAL`, `SERVICIO_PRESTACIONAL` | int / texto | Servicio prestacional (código y nombre). Ejemplos: 56 = CONSULTA EXTERNA, 62 = ATENCIÓN POR EMERGENCIA. |
| `DIAS_HOSP` | float | Días de hospitalización. Vacío cuando no hubo hospitalización (~98.5% de las atenciones). |
| `PRES_ART_SISTOLICA`, `PRES_ART_DIASTOLICA` | float | Presión arterial sistólica y diastólica en mmHg (\*). Vacío cuando no se midió (~49.8%). |
| `TIPO_PERSONAL_SALUD` | texto | Tipo de profesional que realizó la atención (\*), p. ej. MEDICO. |
| `FECATE_POST_FECFED` | texto (SI/NO) | **Significado no documentado.** Por el nombre podría comparar la fecha de atención (`FECATE`) con otra fecha (¿de fallecimiento?); hay que validarlo antes de usarla. |
| `ES_CAPITA` | texto (S/N) | Indicador de atención bajo financiamiento por cápita (\*). |
| `DESTINO_ASEGURADO` | texto | Destino del paciente al terminar la atención: Alta, Citado, Referido, Hospitalizado, **Fallecido**, etc. |

**2. Diagnósticos (uno o más por atención)**

| Columna | Tipo | Descripción |
|---|---|---|
| `CODDIA` | texto | Código CIE-10 sin punto (p. ej. `I10X` = hipertensión esencial, `E119` = diabetes tipo 2 sin complicaciones). |
| `TIPO_DIAGNOSTICO` | texto | `DEFINITIVO`, `REPETIDO` o `PRESUNTIVO`. |

**3. Medicamentos · 4. Procedimientos · 5. Insumos (uno o más por atención)**

| Columna | Tipo | Descripción |
|---|---|---|
| `CODDIA` | texto | Diagnóstico CIE-10 al que se asocia el consumo. |
| `COD_MEDICAMENTO` / `COD_PROCEDIMIIENTO` / `COD_INSUMO` | texto / texto / int | Código del consumo según la tabla. `COD_PROCEDIMIIENTO` se escribe con doble "I" **en el archivo original**. Los dos primeros se leen como texto para no perder ceros a la izquierda (p. ej. `02642`). |
| `CANTIDAD_ENTREGADA` | int | Cantidad entregada o realizada. |
| `VALOR_NETO` | float | Valor neto del consumo (\*), probablemente en soles. |

## Exploración de problemas en el dataset

#### Unidad de análisis: ¿atención o paciente?
Se compara el número de atenciones (filas de FUAS) con el de pacientes distintos. Es necesario saber para decidir si hay que resumir la información: como el objetivo es predecir **por paciente**, la tabla de modelado debe tener una fila por paciente. Además se describe cuántas atenciones tiene cada paciente: la media, la mediana y el máximo muestran cuánto se repite el mismo paciente y si existen casos extremos.

In [7]:
print("Atenciones (filas de FUAS):", f"{len(fuas):,}")
print("Pacientes distintos:", f"{fuas['CODIGO_ANONIMIZADO'].nunique():,}")

atenciones_por_paciente = fuas['CODIGO_ANONIMIZADO'].value_counts()
atenciones_por_paciente.describe().round(1)

Atenciones (filas de FUAS): 24,286,707
Pacientes distintos: 1,026,473


count    1026473.0
mean          23.7
std           26.2
min            1.0
25%            6.0
50%           15.0
75%           33.0
max         1589.0
Name: count, dtype: float64

Se observa que cada paciente acumula muchas atenciones y que la distribución es muy asimétrica (el máximo está muy lejos de la mediana). Si se modelara por atención, **el mismo paciente aparecería varias veces**, incluso repartido entre train y test, lo que produciría fuga de información. Por lo tanto habrá que **resumir todo por paciente** antes de modelar, y conviene revisar los pacientes con un número extremo de atenciones, porque podrían dominar los totales (consumos, costos) si no se normalizan o recortan.

#### Consistencia de los datos del paciente
`SEXO`, `EDAD` y la ubicación se repiten en cada atención. Al resumir por paciente hay que elegir un valor, así que es necesario saber antes si esos datos son coherentes. Si un mismo paciente aparece con sexos distintos, o con una edad que varía más de lo posible en 3 años de datos, hay errores de registro o de anonimización. Esto define cómo resumir al paciente (¿el último valor? ¿el más frecuente?) y cuántos casos dudosos convendría excluir.

In [8]:
por_paciente = fuas.groupby('CODIGO_ANONIMIZADO', observed=True).agg(
    n_sexos=('SEXO', 'nunique'),
    edad_min=('EDAD', 'min'),
    edad_max=('EDAD', 'max'),
    n_departamentos=('DEPARTAMENTO', 'nunique'),
)
rango_edad = por_paciente['edad_max'] - por_paciente['edad_min']
print(f"Pacientes con más de un SEXO: {(por_paciente['n_sexos'] > 1).sum():,}")
print(f"Pacientes cuya edad varía más de 3 años (imposible en 2022-2024): {(rango_edad > 3).sum():,}")
print(f"Pacientes atendidos en más de un departamento: {(por_paciente['n_departamentos'] > 1).sum():,}")
print("\nVariación de edad por paciente (años):")
print(rango_edad.describe(percentiles=[.5, .99]).round(1).to_string())

Pacientes con más de un SEXO: 0
Pacientes cuya edad varía más de 3 años (imposible en 2022-2024): 0
Pacientes atendidos en más de un departamento: 144,968

Variación de edad por paciente (años):
count    1026473.0
mean           1.8
std            1.1
min            0.0
50%            2.0
99%            3.0
max            3.0


**Interpretación**

Se observa que los datos demográficos básicos presentan una consistencia excelente: no existen registros con inconsistencias en el sexo ni variaciones atípicas en la edad (la variación máxima de edad se mantiene estrictamente dentro del rango biológico posible para un período de 3 años). Sin embargo, se aprecia que un grupo relevante de pacientes (144,968) registra atenciones en más de un departamento.   Por lo tanto, no será necesario filtrar o descartar pacientes por inconsistencias graves de identidad o edad, pero sí habrá que definir un criterio para consolidar la ubicación geográfica al momento de reducir las atenciones a pacientes únicos (por ejemplo, seleccionar el departamento de la última atención o el de mayor frecuencia de visita)

#### Fechas
Se requiere conocer el rango de fechas para decidir cómo partir el tiempo en "pasado" (variables X) y "futuro" (objetivo y). Se puede usar `min()` y `max()` directamente porque `FECHA_ATENCION` está en formato AAAAMMDD: ordenar esos números equivale a ordenar fechas.

In [9]:
print("Primera atención:", fuas['FECHA_ATENCION'].min())
print("Última atención:", fuas['FECHA_ATENCION'].max())

Primera atención: 20220101
Última atención: 20241231


### Cobertura y escala observadas

- Se observa que el período cubre tres años calendario completos, lo que permite separar dos años de observación y uno de resultado.
- La fecha de corte de la extracción es posterior a la última atención, por lo tanto el último año no está truncado por el corte.
- `FECHA_CORTE` y `ULTIMO_MES_CONSUMOS` describen la extracción y no la atención, así que habrá que descartarlas como variables.

#### Ventanas de observación y de resultado
Se fijan las fechas que separan las dos ventanas. La ventana de observación (2022-2023) y la de resultado (2024) **no se superponen**: así se evita la fuga de información, es decir, usar el futuro para predecir el futuro. Todas las variables X se calcularán con atenciones hasta `FIN_OBS` y el objetivo con atenciones desde `INI_RES`.

In [10]:
FIN_OBS = 20231231
INI_RES = 20240101

### Cómo se relacionan

```
                                 ┌─ ID_REGISTRO_REL ─► Diagnósticos   (N filas por atención)
FUAS (1 fila por atención) ──────┼─ ID_REGISTRO_REL ─► Medicamentos   (N filas por atención)
 └─ CODIGO_ANONIMIZADO           ├─ ID_REGISTRO_REL ─► Procedimientos (N filas por atención)
    (identifica al paciente)     └─ ID_REGISTRO_REL ─► Insumos        (N filas por atención)
```

- Todas las tablas se unen con `ID_REGISTRO_REL` (el número de la atención). El paciente se identifica con `CODIGO_ANONIMIZADO`.
- **Solo FUAS tiene el paciente y la fecha.** Las otras 4 tablas no los traen, así que **toda unión por paciente o por fecha tiene que pasar por FUAS**.
- Medicamentos, Procedimientos e Insumos incluyen además `CODDIA`: el diagnóstico al que se asocia ese consumo, lo que permitiría atribuir cada consumo a una enfermedad concreta de la atención.
- Las 5 tablas tienen una relación **uno a muchos** con FUAS; unirlas "tal cual" multiplicaría las filas, por lo que cada tabla se debe **resumir antes de unirla**.

#### Llaves y granularidad entre las 5 tablas
Todo el cruce depende de `ID_REGISTRO_REL`. Es necesario saber:

- si identifica **una sola atención** en FUAS: si un ID se repite, "una fila = una atención" sería falso y las uniones duplicarían filas;
- si hay IDs **huérfanos** (presentes en una tabla hija pero no en FUAS): se perderían en un merge interno y quedarían sin paciente ni fecha, porque esos datos solo están en FUAS;
- cuántas atenciones de FUAS **no tienen filas** en cada tabla hija: puede ser normal (una consulta sin medicamentos) y en ese caso no deben tratarse como datos faltantes;
- **cuántas filas por atención** aporta cada tabla: si se unen "tal cual", las filas se multiplican (3 diagnósticos x 5 medicamentos x 8 procedimientos = 120 filas para una sola atención).

In [11]:
n_ids = fuas['ID_REGISTRO_REL'].nunique()
estado = 'OK: 1 fila = 1 atención' if n_ids == len(fuas) else 'HAY IDs REPETIDOS'
print(f"FUAS: {len(fuas):,} filas y {n_ids:,} ID_REGISTRO_REL únicos -> {estado}")

ids_fuas = pd.Index(fuas['ID_REGISTRO_REL'].unique())
for nombre, df in dataframes.items():
    if nombre == 'FUAS':
        continue
    ids_hija = pd.Index(df['ID_REGISTRO_REL'].unique())
    huerfanos = (~ids_hija.isin(ids_fuas)).sum()
    sin_filas = (~ids_fuas.isin(ids_hija)).sum()
    print(f"{nombre}: {huerfanos:,} IDs sin atención en FUAS | "
          f"{sin_filas:,} atenciones de FUAS ({sin_filas / len(ids_fuas):.1%}) sin filas en esta tabla")

FUAS: 24,286,707 filas y 24,286,707 ID_REGISTRO_REL únicos -> OK: 1 fila = 1 atención
Diagnosticos: 0 IDs sin atención en FUAS | 0 atenciones de FUAS (0.0%) sin filas en esta tabla
Medicamentos: 0 IDs sin atención en FUAS | 11,172,353 atenciones de FUAS (46.0%) sin filas en esta tabla
Procedimientos: 0 IDs sin atención en FUAS | 7,352,441 atenciones de FUAS (30.3%) sin filas en esta tabla
Insumos: 0 IDs sin atención en FUAS | 18,957,998 atenciones de FUAS (78.1%) sin filas en esta tabla


In [12]:
for nombre, df in dataframes.items():
    if nombre == 'FUAS':
        continue
    por_atencion = df['ID_REGISTRO_REL'].value_counts()
    print(f"\n{nombre}: filas por atención")
    print(por_atencion.describe(percentiles=[.5, .9, .99]).round(1).to_string())


Diagnosticos: filas por atención
count    24286707.0
mean            1.6
std             0.9
min             1.0
50%             1.0
90%             3.0
99%             5.0
max            10.0

Medicamentos: filas por atención
count    13114354.0
mean            3.0
std             3.3
min             1.0
50%             2.0
90%             5.0
99%            17.0
max           120.0

Procedimientos: filas por atención
count    16934266.0
mean            3.1
std             5.4
min             1.0
50%             1.0
90%             7.0
99%            28.0
max           156.0

Insumos: filas por atención
count    5328709.0
mean           3.0
std            4.8
min            1.0
50%            1.0
90%            6.0
99%           25.0
max          184.0


**Interpretación**

- Se observa que `ID_REGISTRO_REL` es único en FUAS, por lo tanto FUAS puede usarse como tabla base con una fila por atención.
- No hay IDs huérfanos en ninguna tabla hija: todas las filas de diagnósticos y consumos pueden vincularse a un paciente y a una fecha a través de FUAS, y un merge interno no pierde registros.
- Toda atención tiene al menos un diagnóstico, pero una parte importante de las atenciones no tiene medicamentos, procedimientos o insumos. Esa ausencia significa "no hubo consumo", por lo tanto al agregar habrá que rellenar con 0 y no con `NaN`.
- Todas las tablas hijas aportan varias filas por atención, con colas largas en los consumos. Por lo tanto habrá que **resumir cada tabla por atención o por paciente antes de unirla** a FUAS; unirlas directamente multiplicaría las filas y distorsionaría cualquier conteo o suma.

#### Valores faltantes
Se cuenta cuántos valores vacíos tiene cada columna de FUAS. Es necesario saber para distinguir entre un dato perdido (que habría que imputar o descartar) y un vacío que tiene significado propio (que habría que recodificar).

In [13]:
missing = fuas.isnull().sum()
missing_pct = (missing / len(fuas)) * 100

missing_df = pd.DataFrame({
    'Columna': missing.index,
    'Faltantes': missing.values,
    'Porcentaje (%)': missing_pct.values.round(2)
})
missing_df[missing_df['Faltantes'] > 0].sort_values('Faltantes', ascending=False)

,Columna,Faltantes,Porcentaje (%)
14,DIAS_HOSP,23927857,98.52
15,PRES_ART_SISTOLICA,12081688,49.75
16,PRES_ART_DIASTOLICA,12081688,49.75


**Interpretación**

- Solo tres columnas tienen vacíos. `DIAS_HOSP` falta cuando **no hubo hospitalización**: no es un dato perdido sino 0 días, por lo tanto habrá que rellenarlo con 0 (y su presencia puede usarse como indicador de hospitalización).
- La presión arterial falta cuando **no se midió** en esa atención; sistólica y diastólica faltan exactamente en las mismas filas, lo que indica que se registran juntas. No conviene imputarla por atención: al resumir por paciente se usarán las mediciones disponibles (promedio, máximo, última) y el número de mediciones, y habrá que revisar si el vacío depende del tipo de servicio (ver Calidad de las variables numéricas).

#### Búsqueda de filas duplicadas
Las filas idénticas repetidas inflan los conteos (de diagnósticos, de procedimientos) y pueden distorsionar las variables que resumen cantidades. Es necesario medirlas en las 5 tablas antes de decidir si se eliminan.

Nota: `duplicated()` sobre tablas de decenas de millones de filas puede agotar la memoria (en la última ejecución terminó con `MemoryError`). Se conserva la celda; si vuelve a fallar, conviene ejecutarla tabla por tabla, liberando memoria entre una y otra.

In [14]:
for nombre, df in dataframes.items():
    print(f"{nombre}: {df.duplicated().sum():,} filas duplicadas")

FUAS: 0 filas duplicadas
Diagnosticos: 4,489 filas duplicadas
Medicamentos: 0 filas duplicadas
Procedimientos: 7,836 filas duplicadas
Insumos: 0 filas duplicadas


#### Duplicados: ¿errores o repeticiones legítimas?
Una fila repetida en un procedimiento puede ser real (el mismo examen hecho dos veces en la misma atención), así que **no conviene eliminarlas automáticamente**. Es necesario saber cómo se distribuyen para decidir con evidencia: se miran ejemplos y en cuántas atenciones ocurren. Si están concentradas en pocas atenciones podría tratarse de un error de carga; si están repartidas, suelen ser repeticiones reales.

In [15]:
for nombre in ['Diagnosticos', 'Procedimientos']:
    df = dataframes[nombre]
    duplicadas = df[df.duplicated(keep=False)]
    print(f"\n{nombre}: {len(duplicadas):,} filas involucradas en {duplicadas['ID_REGISTRO_REL'].nunique():,} atenciones distintas")
    display(duplicadas.sort_values('ID_REGISTRO_REL').head(6))


Diagnosticos: 8,978 filas involucradas en 3,959 atenciones distintas


,FECHA_CORTE,ULTIMO_MES_CONSUMOS,ID_REGISTRO_REL,CODDIA,TIPO_DIAGNOSTICO
21403395,20250829,202412,23670523,J459,DEFINITIVO
874470,20250829,202412,23670523,J459,DEFINITIVO
11173669,20250829,202412,23670526,K359,DEFINITIVO
11173667,20250829,202412,23670526,K359,DEFINITIVO
874512,20250829,202412,23672192,A09X,DEFINITIVO
874510,20250829,202412,23672192,A09X,DEFINITIVO



Procedimientos: 15,534 filas involucradas en 7,649 atenciones distintas


,FECHA_CORTE,ULTIMO_MES_CONSUMOS,ID_REGISTRO_REL,CODDIA,COD_PROCEDIMIIENTO,CANTIDAD_ENTREGADA,VALOR_NETO
18467580,20250829,202412,23470830,A64X,99402.05,1,0.0
18467579,20250829,202412,23470830,A64X,99402.05,1,0.0
26879069,20250829,202412,23472119,Z017,81000,1,0.0
46112679,20250829,202412,23472119,Z017,81000,1,0.0
52813938,20250829,202412,23472461,R104,81000,1,0.0
51639194,20250829,202412,23472461,R104,81000,1,0.0


**Interpretación**

Se observa que las filas completamente idénticas se concentran en un número muy reducido de atenciones (alrededor de 4,000 en diagnósticos y 7,600 en procedimientos, una proporción ínfima dentro de las más de 24 millones de atenciones totales).   En Diagnósticos, registrar exactamente el mismo código CIE-10 y tipo en una misma atención representa una redundancia clara del sistema que inflaría los conteos de antecedentes. En Procedimientos, aunque se pueden realizar múltiples exámenes en una sola consulta, el hecho de que coincidan idénticamente el código, la cantidad entregada y el monto sugiere duplicaciones puntuales de registro al momento de la carga.   Por lo tanto, habrá que aplicar una deduplicación explícita (drop_duplicates()) sobre las tablas de diagnósticos y procedimientos durante la etapa de limpieza, garantizando que estas líneas repetidas no distorsionen la agregación de características ni el cálculo final de consumos por paciente.

#### Valores imposibles
`describe()` resume mínimo, máximo y cuartiles de cada columna numérica. Es la forma más rápida de detectar valores imposibles (edades o presiones fuera de rango) y columnas constantes (desviación = 0), que no aportan información. En las columnas que son identificadores (`ID_REGISTRO_REL`, `UBIGEO`, `CODIGO_SERV_PRESTACIONAL`) la media o la desviación no significan nada; en ellas solo importa el rango.

In [16]:
fuas.describe().round(1)

,FECHA_CORTE,ULTIMO_MES_CONSUMOS,ID_REGISTRO_REL,FECHA_ATENCION,EDAD,UBIGEO,NIVEL_EESS,CODIGO_SERV_PRESTACIONAL,DIAS_HOSP,PRES_ART_SISTOLICA,PRES_ART_DIASTOLICA
count,24286707.0,24286707.0,24286707.0,24286707.0,24286707.0,24286707.0,24286707.0,24286707.0,358850.0,12205019.0,12205019.0
mean,20250829.0,202412.0,89143758.7,20231706.1,63.9,131222.6,1.6,136.1,9.7,120.0,70.7
std,0.0,0.0,65449523.9,8056.9,14.5,56999.7,0.8,244.7,19.0,20.6,11.3
min,20250829.0,202412.0,23457863.0,20220101.0,14.0,10101.0,0.0,8.0,1.0,50.0,40.0
25%,20250829.0,202412.0,65195997.0,20221123.0,55.0,81208.0,1.0,56.0,3.0,101.0,60.0
50%,20250829.0,202412.0,84858558.0,20230825.0,66.0,150106.0,1.0,56.0,5.0,120.0,70.0
75%,20250829.0,202412.0,100184281.5,20240507.0,75.0,150142.0,2.0,63.0,11.0,130.0,80.0
max,20250829.0,202412.0,532518903.0,20241231.0,119.0,250401.0,3.0,911.0,1467.0,400.0,248.0


**Interpretación**

- Se observa que `FECHA_CORTE` y `ULTIMO_MES_CONSUMOS` tienen desviación nula: son constantes, por lo tanto se descartarán.
- `EDAD` tiene un mínimo por debajo de la mayoría de edad y un máximo poco plausible; habrá que definir un rango válido y revisar cuántas atenciones quedan fuera.
- La presión arterial tiene extremos fisiológicamente improbables en ambos sentidos (máximos de sistólica y diastólica muy por encima de cualquier crisis hipertensiva registrada habitualmente), lo que sugiere errores de digitación. Habrá que establecer umbrales y tratar los valores fuera de rango como faltantes.
- `DIAS_HOSP` presenta un máximo de varios años de hospitalización, imposible dentro de un período de tres años; será necesario recortarlo o marcarlo como error.
- Los cuartiles de las variables clínicas sí son razonables, así que los problemas se concentran en las colas. Los umbrales concretos se evalúan en *Calidad de las variables numéricas*.

### Montos y cantidades entregadas
En las tres tablas de consumos se buscan valores extremos. Se compara la mediana (lo típico) con el máximo: si el máximo es de millones y la mediana es de un dígito, hay cantidades o montos que casi seguro son errores de digitación y que distorsionarían cualquier suma o promedio por paciente.

In [17]:
for nombre, df in [('Medicamentos', medicamentos), ('Procedimientos', procedimientos), ('Insumos', insumos)]:
    print(nombre)
    print(df[['CANTIDAD_ENTREGADA', 'VALOR_NETO']].describe().round(1).loc[['50%', 'max']])
    print()

Medicamentos
     CANTIDAD_ENTREGADA  VALOR_NETO
50%                10.0         0.0
max           7377000.0   1212000.0

Procedimientos
     CANTIDAD_ENTREGADA  VALOR_NETO
50%                 1.0         3.5
max           1501004.0    334461.6

Insumos
     CANTIDAD_ENTREGADA  VALOR_NETO
50%                 1.0         0.2
max            627749.0    184512.0



**Interpretación**

- En las tres tablas el máximo está varios órdenes de magnitud por encima de la mediana, tanto en cantidad como en monto. Por lo tanto, sumar consumos por paciente sin tratar estas colas produciría variables dominadas por unos pocos registros erróneos.
- En Medicamentos la mediana de `VALOR_NETO` es 0: buena parte de las entregas no está valorizada. Habrá que decidir si el 0 es un valor legítimo o un faltante disfrazado antes de usar el monto como variable; el conteo de entregas puede ser más confiable que el monto.
- Será necesario recortar (winsorizar) o excluir los extremos con un criterio basado en percentiles, que se examina en la siguiente sección.

#### Calidad de las variables numéricas
Ya se sabe que hay valores imposibles; ahora es necesario decidir **dónde cortar**. Los percentiles muestran dónde empiezan los extremos y si son casos aislados (errores de digitación) o una cola real; los conteos indican cuántas filas se verían afectadas. Los umbrales usados (250, 150, 365 días, 18 años) son solo valores de referencia para dimensionar el problema, no reglas definitivas.

También se revisa si la falta de presión arterial depende del tipo de servicio: si casi nunca se mide en ciertos servicios, el vacío **no es aleatorio** y ese patrón puede servir como variable.

En la segunda celda se estudian los consumos: la mediana de `VALOR_NETO` en Medicamentos es 0 y hay cantidades de millones, así que es necesario saber si el 0 es un valor "normal" (entrega sin valorizar) o un faltante disfrazado, y si existen cantidades o montos negativos o en cero que no tengan sentido.

In [18]:
pa = fuas[['PRES_ART_SISTOLICA', 'PRES_ART_DIASTOLICA']].dropna()
print("Percentiles de presión arterial (mmHg):")
print(pa.quantile([.001, .01, .5, .99, .999]).to_string())
print(f"\nSistólica <= diastólica (posible inversión o error): {(pa['PRES_ART_SISTOLICA'] <= pa['PRES_ART_DIASTOLICA']).sum():,}")
print(f"Sistólica > 250: {(pa['PRES_ART_SISTOLICA'] > 250).sum():,} | Diastólica > 150: {(pa['PRES_ART_DIASTOLICA'] > 150).sum():,}")

print("\nDIAS_HOSP (solo atenciones con hospitalización):")
print(fuas['DIAS_HOSP'].describe(percentiles=[.5, .9, .99, .999]).round(1).to_string())
print(f"DIAS_HOSP > 365: {(fuas['DIAS_HOSP'] > 365).sum():,}")

print(f"\nEDAD < 18: {(fuas['EDAD'] < 18).sum():,} | EDAD > 100: {(fuas['EDAD'] > 100).sum():,}")

top_servicios = fuas['SERVICIO_PRESTACIONAL'].value_counts().head(8).index
print("\n% de atenciones SIN presión arterial, por servicio (8 más frecuentes):")
print(fuas[fuas['SERVICIO_PRESTACIONAL'].isin(top_servicios)]
      .groupby('SERVICIO_PRESTACIONAL')['PRES_ART_SISTOLICA']
      .apply(lambda s: s.isna().mean() * 100).round(1).sort_values().to_string())

Percentiles de presión arterial (mmHg):
       PRES_ART_SISTOLICA  PRES_ART_DIASTOLICA
0.001                78.0                 40.0
0.010                90.0                 50.0
0.500               120.0                 70.0
0.990               180.0                100.0
0.999               211.0                120.0

Sistólica <= diastólica (posible inversión o error): 0
Sistólica > 250: 1,098 | Diastólica > 150: 1,339

DIAS_HOSP (solo atenciones con hospitalización):
count    358850.0
mean          9.7
std          19.0
min           1.0
50%           5.0
90%          20.0
99%          65.0
99.9%       180.0
max        1467.0
DIAS_HOSP > 365: 165

EDAD < 18: 15,484 | EDAD > 100: 8,752

% de atenciones SIN presión arterial, por servicio (8 más frecuentes):
SERVICIO_PRESTACIONAL
ATENCIÓN INTEGRAL DE SALUD DEL ADULTO MAYOR                        0.7
CONSULTA EXTERNA                                                  29.1
ATENCIÓN POR EMERGENCIA                                          

In [19]:
for nombre in ['Medicamentos', 'Procedimientos', 'Insumos']:
    df = dataframes[nombre]
    print(f"\n{nombre}")
    print(f"  VALOR_NETO == 0: {(df['VALOR_NETO'] == 0).mean():.1%} | VALOR_NETO < 0: {(df['VALOR_NETO'] < 0).sum():,}")
    print(f"  CANTIDAD_ENTREGADA <= 0: {(df['CANTIDAD_ENTREGADA'] <= 0).sum():,}")
    print(df[['CANTIDAD_ENTREGADA', 'VALOR_NETO']].quantile([.5, .9, .99, .999]).round(1).to_string())


Medicamentos
  VALOR_NETO == 0: 52.4% | VALOR_NETO < 0: 0
  CANTIDAD_ENTREGADA <= 0: 0
       CANTIDAD_ENTREGADA  VALOR_NETO
0.500                10.0         0.0
0.900                60.0        11.2
0.990               120.0       136.1
0.999             18360.0      1077.3

Procedimientos
  VALOR_NETO == 0: 38.7% | VALOR_NETO < 0: 0
  CANTIDAD_ENTREGADA <= 0: 0
       CANTIDAD_ENTREGADA  VALOR_NETO
0.500                 1.0         3.5
0.900                 2.0        20.3
0.990                16.0       257.8
0.999                95.0      1045.5

Insumos
  VALOR_NETO == 0: 47.6% | VALOR_NETO < 0: 0
  CANTIDAD_ENTREGADA <= 0: 0
       CANTIDAD_ENTREGADA  VALOR_NETO
0.500                 1.0         0.2
0.900                 7.0        16.5
0.990                70.0       236.2
0.999               301.0      1693.4


#### Categorías importantes
Se revisan las variables categóricas que condicionan decisiones de limpieza y la construcción del objetivo:
- `TIPO_DIAGNOSTICO`: define qué diagnósticos son confiables (definitivos y repetidos) y cuáles se descartan (presuntivos).
- `DESTINO_ASEGURADO`: contiene la categoría `'Fallecido'`, que es una de las complicaciones del objetivo.

Nota: el ancho de la salida de `TIPO_DIAGNOSTICO` sugiere que los valores tienen espacios en blanco al final; por eso, en las secciones de Comorbilidades y Variable objetivo se limpian con `.str.strip()` antes de filtrar.

In [20]:
print(diagnosticos['TIPO_DIAGNOSTICO'].value_counts())
print()
print(fuas['DESTINO_ASEGURADO'].value_counts())

TIPO_DIAGNOSTICO
DEFINITIVO         27447663
REPETIDO            7299335
PRESUNTIVO          3332015
Name: count, dtype: int64

DESTINO_ASEGURADO
Citado                   14228018
Alta                      8417925
Sin Destino                965658
Contrarreferido            341907
Ref. Consulta Externa      256201
Fallecido                   31269
Ref. Emergencia             22295
Ref. Apoyo al Diax          15899
Hospitalizado                7050
Corte Administrativo          485
Name: count, dtype: int64


**Interpretación**

- Se observa que la gran mayoría de los diagnósticos son definitivos o repetidos; los presuntivos son una fracción menor. Por lo tanto, quedarse solo con `DEFINITIVO` y `REPETIDO` reduce el ruido sin perder la mayor parte de la información.
- En `DESTINO_ASEGURADO` la categoría `'Fallecido'` es poco frecuente, lo que anticipa una clase positiva pequeña si se usara sola como objetivo. También existen categorías residuales (`Sin Destino`, `Corte Administrativo`) que habrá que agrupar o tratar como desconocidas.

#### Diagnósticos más frecuentes (`CODDIA`)
Es necesario saber qué códigos CIE-10 dominan. Si pocos códigos concentran la mayoría de los registros (el acumulado crece rápido), se podría trabajar con los más frecuentes o agruparlos, en vez de crear una columna por cada uno de los miles de códigos posibles. También sirve para confirmar que la HTA (`I10X`) es el diagnóstico más común, como se espera en este dataset. Se muestran solo los 15 primeros, que alcanzan para ver la concentración.

In [21]:
dist = pd.DataFrame({
    'Conteo': diagnosticos['CODDIA'].value_counts(),
    'Porcentaje': (diagnosticos['CODDIA'].value_counts(normalize=True) * 100).round(2),
    'Acumulado %': (diagnosticos['CODDIA'].value_counts(normalize=True) * 100).cumsum().round(2)
})

display(dist.head(15))

,Conteo,Porcentaje,Acumulado %
CODDIA,,,
I10X,5118712,13.44,13.44
E119,1566644,4.11,17.56
Z017,1383025,3.63,21.19
N390,1039254,2.73,23.92
Z133,832200,2.19,26.10
M545,721308,1.89,28.00
Z108,675751,1.77,29.77
E785,663627,1.74,31.51
K297,635433,1.67,33.18


**Interpretación**

- Se confirma que `I10X` (hipertensión esencial) es el diagnóstico más frecuente, coherente con la definición del dataset.
- Entre los primeros aparecen diagnósticos que forman parte de las comorbilidades y complicaciones de interés (`E119` diabetes tipo 2, `E785` hiperlipidemia), lo que indica que habrá suficiente señal para construir esos indicadores.
- Varios códigos frecuentes son del capítulo Z (controles, exámenes, factores que influyen en el estado de salud), que describen el tipo de contacto y no una enfermedad; habrá que decidir si se excluyen o se agrupan aparte.
- El acumulado crece lentamente después de los primeros códigos: la distribución tiene una cola larga, por lo tanto no es viable una columna por código y habrá que agrupar (por prefijo de 3 caracteres, capítulo CIE-10 o grupos clínicos).

#### Categorías y códigos
Se revisan todas las categorías de FUAS (incluida `FECATE_POST_FECFED`, cuyo significado no está documentado) y cuántos códigos distintos hay en cada tabla. Es necesario saber para:
- evitar sorpresas al codificar (valores raros, categorías casi vacías);
- comprobar que el código y el nombre del servicio se corresponden 1 a 1;
- cruzar `FECATE_POST_FECFED` con `'Fallecido'` para entender qué significa esa columna: si "SI" coincide con atenciones de fallecidos o con atenciones posteriores a un fallecimiento, es una señal de calidad de datos y de posible fuga de información.

En la segunda celda se mide la concentración de códigos de diagnóstico, medicamento, procedimiento e insumo: cada código sería una columna potencial y, si unos pocos concentran el uso, conviene quedarse con los más frecuentes o agrupar (capítulo CIE-10, grupo terapéutico). Los archivos solo traen códigos, sin nombres; para agrupar medicamentos y procedimientos hará falta un catálogo externo (por ejemplo, para identificar los antihipertensivos).

In [22]:
es_fallecido = fuas['DESTINO_ASEGURADO'].str.strip() == 'Fallecido'

for col in ['SEXO', 'NIVEL_EESS', 'TIPO_PERSONAL_SALUD', 'ES_CAPITA', 'FECATE_POST_FECFED']:
    print(fuas[col].value_counts(dropna=False).to_string(), "\n")

print("Servicios prestacionales -> códigos únicos:", fuas['CODIGO_SERV_PRESTACIONAL'].nunique(),
      "| nombres únicos:", fuas['SERVICIO_PRESTACIONAL'].nunique())
print("\nFECATE_POST_FECFED vs destino 'Fallecido':")
print(pd.crosstab(fuas['FECATE_POST_FECFED'], es_fallecido, rownames=['FECATE_POST_FECFED'], colnames=['Fallecido']))

SEXO
FEMENINO     16086212
MASCULINO     8200495 

NIVEL_EESS
1    14764218
3     4903153
2     4591449
0       27887 

TIPO_PERSONAL_SALUD
MEDICO                                     16681547
PSICOLOGO                                   1529761
TEC. ENFERMERIA                             1145856
TECNOLOGO MEDICO                            1143291
ENFERMERO(A)                                 931785
NUTRICIONISTA                                669977
BIOLOGO                                      661137
AUXILIAR DE ENFERMERIA                       612730
OBSTETRIZ                                    488327
FARMACEUTICO                                 247404
ODONTOLOGO                                    85858
TECNICO LABORATORIO                           43178
TRABAJADORA SOCIAL                            20247
OTROS                                         13901
TECNICO EN RAYOS X                             5019
TECNICO DE FARMACIA                            4401
TEC ENFERMERIA PROG RADIOLOG

In [23]:
codigos = [('Diagnosticos', 'CODDIA'), ('Medicamentos', 'COD_MEDICAMENTO'),
           ('Procedimientos', 'COD_PROCEDIMIIENTO'), ('Insumos', 'COD_INSUMO')]
for nombre, col in codigos:
    frecuencias = dataframes[nombre][col].value_counts(normalize=True)
    print(f"{nombre}.{col}: {len(frecuencias):,} códigos distintos | "
          f"top 20 cubren {frecuencias.head(20).sum():.1%} | top 100: {frecuencias.head(100).sum():.1%} | "
          f"top 500: {frecuencias.head(500).sum():.1%}")

Diagnosticos.CODDIA: 10,422 códigos distintos | top 20 cubren 43.5% | top 100: 67.6% | top 500: 88.5%
Medicamentos.COD_MEDICAMENTO: 6,063 códigos distintos | top 20 cubren 48.0% | top 100: 84.1% | top 500: 99.2%
Procedimientos.COD_PROCEDIMIIENTO: 4,760 códigos distintos | top 20 cubren 42.0% | top 100: 80.1% | top 500: 98.3%
Insumos.COD_INSUMO: 16,271 códigos distintos | top 20 cubren 51.4% | top 100: 74.6% | top 500: 91.4%


**Interpretación**

*(Pendiente de ejecución: completar con las observaciones a partir de la salida real de la celda anterior.)*

#### Comorbilidades en la ventana de observación
Las comorbilidades metabólicas registradas **antes** del período de resultado son candidatas naturales a variables predictoras. Es necesario saber cuántos pacientes las presentan en 2022-2023 y cuántas acumula cada uno, para decidir si se incluyen como indicadores individuales y como recuento.

| Comorbilidad | Códigos CIE-10 (prefijo de 3 caracteres) |
|---|---|
| Diabetes | `E10`-`E14` |
| Dislipidemia | `E78` |
| Obesidad | `E66` |
| Otros trastornos metabólicos / síndrome metabólico | `E88` (incluye `E88.9` / `E889`) |

Criterios:
- Solo diagnósticos `DEFINITIVO` y `REPETIDO` (limpiados con `.str.strip()`).
- Se compara el prefijo de 3 caracteres del código, ignorando puntos (`E119` -> `E11`).
- Se calcula sobre los **pacientes únicos con al menos una atención en 2022-2023**; los pacientes sin el diagnóstico quedan en 0.
- Se generan un indicador por comorbilidad (`COMORB_*`) y un recuento discreto por paciente (`N_COMORBILIDADES`, de 0 a 4).

In [24]:
COMORBILIDADES = {
    'DIABETES': ('E10', 'E11', 'E12', 'E13', 'E14'),
    'DISLIPIDEMIA': ('E78',),
    'OBESIDAD': ('E66',),
    'OTROS_METABOLICOS': ('E88',),
}
prefijo_a_comorb = {p: g for g, prefijos in COMORBILIDADES.items() for p in prefijos}

tipo_valido = diagnosticos['TIPO_DIAGNOSTICO'].str.strip().isin(['DEFINITIVO', 'REPETIDO'])
dx_validos = diagnosticos.loc[tipo_valido, ['ID_REGISTRO_REL', 'CODDIA']]
cod = dx_validos['CODDIA'].str.strip().str.replace('.', '', regex=False).str.upper().str[:3]

dx_comorb = pd.DataFrame({
    'ID_REGISTRO_REL': dx_validos['ID_REGISTRO_REL'],
    'COMORBILIDAD': cod.map(prefijo_a_comorb),
}).dropna(subset=['COMORBILIDAD'])

atenciones_obs = fuas.loc[fuas['FECHA_ATENCION'] <= FIN_OBS, ['ID_REGISTRO_REL', 'CODIGO_ANONIMIZADO']]
pacientes_obs = pd.Index(atenciones_obs['CODIGO_ANONIMIZADO'].unique(), name='CODIGO_ANONIMIZADO')

comorb_obs = (dx_comorb.merge(atenciones_obs, on='ID_REGISTRO_REL')
              [['CODIGO_ANONIMIZADO', 'COMORBILIDAD']].drop_duplicates())

comorbilidades = (pd.crosstab(comorb_obs['CODIGO_ANONIMIZADO'], comorb_obs['COMORBILIDAD'])
                  .clip(upper=1)
                  .reindex(index=pacientes_obs, columns=list(COMORBILIDADES), fill_value=0)
                  .add_prefix('COMORB_'))
comorbilidades['N_COMORBILIDADES'] = comorbilidades.sum(axis=1)

print(f"Pacientes con atenciones en 2022-2023: {len(pacientes_obs):,}")
display(pd.DataFrame({
    'Pacientes': comorbilidades.filter(like='COMORB_').sum(),
    '% de pacientes': (comorbilidades.filter(like='COMORB_').mean() * 100).round(2),
}))
display(comorbilidades['N_COMORBILIDADES'].value_counts().sort_index()
        .rename_axis('N_COMORBILIDADES').to_frame('Pacientes'))

Pacientes con atenciones en 2022-2023: 939,664


,Pacientes,% de pacientes
COMORBILIDAD,,
COMORB_DIABETES,211946,22.56
COMORB_DISLIPIDEMIA,271518,28.90
COMORB_OBESIDAD,176080,18.74
COMORB_OTROS_METABOLICOS,3660,0.39


,Pacientes
N_COMORBILIDADES,
0,496963
1,263144
2,139232
3,39704
4,621


**Interpretación**

Existe una correspondencia perfecta 1 a 1 entre los códigos y los nombres de los servicios prestacionales (50 códigos para 50 nombres), mientras que atributos como SEXO y ES_CAPITA presentan categorías binarias limpias y bien definidas
Aunque existen miles de códigos distintos en las tablas hijas (hasta 16,271 en insumos y 10,422 en diagnósticos), la masa del dataset se concentra en pocos ítems: el top 100 a top 500 de códigos cubre entre el 80% y el 99% de todas las atenciones y consumos del SIS
Para la limpieza implicaría:
- Eliminar redundancias y columnas ambiguas: Descartar una de las dos columnas redundantes de servicio prestacional (CODIGO_SERV_PRESTACIONAL o SERVICIO_PRESTACIONAL) y omitir la variable FECATE_POST_FECFED por no aportar señal predictiva ni servir como proxy de fallecimiento (manteniendo DESTINO_ASEGURADO == 'Fallecido' como único criterio).
- Agrupar categorías marginales: Colapsar las categorías de baja frecuencia en TIPO_PERSONAL_SALUD (ej. técnicos en saneamiento, personal administrativo) y niveles raros en NIVEL_EESS (nivel 0) bajo una etiqueta común como 'OTROS'.
- Controlar la dimensionalidad de códigos: Evitar la creación de miles de columnas dispersas (one-hot encoding masivo). En su lugar, habrá que filtrar únicamente los top $N$ códigos más frecuentes (por ejemplo, el top 100 o top 500) o agrupar por categorías superiores (capítulos CIE-10 para diagnósticos o grupos terapéuticos mediante catálogos externos para medicamentos) antes de pivotar y resumir por paciente.   

#### Construcción de la variable objetivo: ¿cuántas complicaciones hay?
Antes de limpiar es necesario saber **cuántos pacientes** tienen cada complicación, **cuántos ya la tenían en 2022-2023** y **qué tan desbalanceada** queda la clase positiva. Esta sección integra los hallazgos previos: la llave `ID_REGISTRO_REL` para llevar el diagnóstico a paciente y fecha, el filtro de `TIPO_DIAGNOSTICO` con espacios limpiados y la categoría `'Fallecido'` de `DESTINO_ASEGURADO`.

Criterios:
- 9 grupos de diagnósticos (ver tabla del Objetivo) más el fallecimiento.
- Solo diagnósticos `DEFINITIVO` y `REPETIDO`, limpiados con `.str.strip()`.
- Comparación por prefijo de 3 caracteres ignorando puntos (`E119` -> `E11`).
- Para cada grupo se cuenta a los pacientes con el evento en 2022-2023, en 2024 y los **casos nuevos** (en 2024 y no antes). Es necesario distinguirlos para decidir si los pacientes con el evento previo se excluyen (predecir el **primer** evento) o se usan como variable.

Los grupos son una propuesta inicial que debe validarse con criterio clínico. Diabetes, dislipidemia y obesidad también aparecen como comorbilidades: si se mantienen en el objetivo, sus indicadores de 2022-2023 explican de forma casi directa el resultado, lo que habrá que evaluar en la interpretación.

In [25]:
COMPLICACIONES = {
    'ACV / AIT': ('I60', 'I61', 'I62', 'I63', 'I64', 'G45'),
    'Infarto / IAM': ('I21', 'I22', 'I23'),
    'Insuficiencia cardíaca': ('I50',),
    'Enfermedad renal': ('N17', 'N18', 'N19'),
    'Diabetes': ('E10', 'E11', 'E12', 'E13', 'E14'),
    'Dislipidemia': ('E78',),
    'Obesidad': ('E66',),
    'Fibrilación auricular': ('I48',),
    'Cardiopatía isquémica': ('I20', 'I24', 'I25'),
}
prefijo_a_grupo = {p: g for g, prefijos in COMPLICACIONES.items() for p in prefijos}

tipo_valido = diagnosticos['TIPO_DIAGNOSTICO'].str.strip().isin(['DEFINITIVO', 'REPETIDO'])
dx_validos = diagnosticos.loc[tipo_valido, ['ID_REGISTRO_REL', 'CODDIA']]
cod = dx_validos['CODDIA'].str.strip().str.replace('.', '', regex=False).str.upper().str[:3]

atenciones = fuas[['ID_REGISTRO_REL', 'CODIGO_ANONIMIZADO', 'FECHA_ATENCION']]
es_fallecido = fuas['DESTINO_ASEGURADO'].str.strip() == 'Fallecido'

eventos_dx = (pd.DataFrame({'ID_REGISTRO_REL': dx_validos['ID_REGISTRO_REL'], 'GRUPO': cod.map(prefijo_a_grupo)})
              .dropna(subset=['GRUPO'])
              .merge(atenciones, on='ID_REGISTRO_REL'))
eventos_fallecido = atenciones[es_fallecido.values].assign(GRUPO='Fallecimiento')
eventos = pd.concat([eventos_dx, eventos_fallecido], ignore_index=True)

def pacientes_por_ventana(eventos_grupo):
    en_obs = set(eventos_grupo.loc[eventos_grupo['FECHA_ATENCION'] <= FIN_OBS, 'CODIGO_ANONIMIZADO'].unique())
    en_res = set(eventos_grupo.loc[eventos_grupo['FECHA_ATENCION'] >= INI_RES, 'CODIGO_ANONIMIZADO'].unique())
    return en_obs, en_res

resultados = {grupo: pacientes_por_ventana(eventos[eventos['GRUPO'] == grupo])
              for grupo in list(COMPLICACIONES) + ['Fallecimiento']}

obs_any = set().union(*[o for o, _ in resultados.values()])
res_any = set().union(*[r for _, r in resultados.values()])
resultados['CUALQUIER complicación'] = (obs_any, res_any)

resumen = pd.DataFrame({
    nombre: {
        'Pacientes en 2022-2023': len(o),
        'Pacientes en 2024': len(r),
        'En 2024 y NO antes (casos nuevos)': len(r - o),
    } for nombre, (o, r) in resultados.items()
}).T
display(resumen)

pacientes_2024 = fuas.loc[fuas['FECHA_ATENCION'] >= INI_RES, 'CODIGO_ANONIMIZADO'].nunique()
print(f"Casos nuevos en 2024 / pacientes con atenciones en 2024: {len(res_any - obs_any) / max(pacientes_2024, 1):.2%}")

,Pacientes en 2022-2023,Pacientes en 2024,En 2024 y NO antes (casos nuevos)
ACV / AIT,28544,18245,13838
Infarto / IAM,4912,3218,2980
Insuficiencia cardíaca,26344,18721,12569
Enfermedad renal,42662,35576,20536
Diabetes,211946,170620,50627
Dislipidemia,271518,194757,92154
Obesidad,176080,110508,63051
Fibrilación auricular,13072,9755,5281
Cardiopatía isquémica,18915,13705,9651
Fallecimiento,19357,11847,11846


Casos nuevos en 2024 / pacientes con atenciones en 2024: 15.52%
